# Per-Airline Flight Count vs. Stock Price: Daily & Earnings-Event Analysis

The companion `flight_volume_vs_airline_stocks.ipynb` notebook used TSA checkpoint volume — an
airline-agnostic, aggregate measure of *all* U.S. air travel demand. This notebook asks a more
targeted question: does **each airline's own flight count** (how many flights *that specific
carrier* actually operated) relate to **that airline's own stock price**, either day-to-day or
around its quarterly earnings announcements?

**Data source**: the U.S. DOT/BTS **Airline On-Time Performance** dataset
(`transtats.bts.gov`) — one row per scheduled mainline flight, free, no API key, published monthly.
We aggregate raw flight records into daily flight counts per carrier ourselves.

**Three important limitations to understand before trusting any result below** (unlike the TSA/
Wikipedia notebooks, which used near-real-time data):

1. **Reporting lag**: BTS publishes each month's data roughly 2-3 months after the fact — as of
   this run, the most recent available month is 2-3 months behind today. That means this dataset
   **cannot function as a live, same-day trading signal** the way TSA/Wikipedia data could. It's
   useful for testing *whether a relationship exists at all* — which is exactly what firms that pay
   for real-time ADS-B flight-tracking data (Cirium, FlightAware, OAG) are trying to monetize by
   removing this lag.
2. **Mainline-only, not total network**: we filter to each carrier's own reporting code (`DL`, `UA`,
   `AA`, `WN` for Delta/United/American/Southwest). Southwest has no regional subsidiary, so its
   mainline count ≈ its entire network. Delta/United/American each operate ~40-50% of their total
   departures under separate regional-carrier codes (Endeavor, SkyWest, Envoy, etc.), so their
   mainline-only counts systematically understate their true network size. **Only compare each
   carrier's trend to itself over time — never compare raw levels across carriers.**
3. **Heavy raw data**: each month's raw file is ~250MB uncompressed (one row per individual flight).
   We download, aggregate to a few hundred rows per month, and discard the raw data immediately —
   aggregated results are cached locally in `data_cache/` so re-running this notebook doesn't
   re-download everything.

**Hypotheses:**
1. Growth in a carrier's own flight count correlates with that carrier's own stock returns.
2. Given the lag, "stock leads flight data" is the more practically interesting direction — it
   tells us whether the market already prices in capacity changes before BTS confirms them.
3. Flight-count trends **already published** by the time an airline reports earnings might predict
   that quarter's EPS surprise or earnings-day stock reaction — a genuine trading-relevant test,
   distinct from the (also interesting, but not tradeable in real time) contemporaneous relationship.

> ⚠️ Research only, not investment advice. The earnings-event analysis in particular has a small
> sample size (roughly one data point per carrier per quarter) — treat it as exploratory, not proof.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install yfinance pandas numpy matplotlib seaborn scipy statsmodels requests lxml

import warnings
warnings.filterwarnings("ignore")

import os
import io
import time
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)

CACHE_DIR = "data_cache"
os.makedirs(CACHE_DIR, exist_ok=True)


## 1. Data Collection

**Flight counts**: for each month, download the BTS On-Time Performance ZIP, read only the
`FlightDate`/`Reporting_Airline` columns (to keep memory small), filter to mainline codes, and
collapse to one row per (date, carrier) with a flight count. The full per-flight data is discarded
immediately after aggregating; only the small aggregated table is cached to disk.

`START_YEAR`/`START_MONTH` controls how much history to pull — the default below (~24 months) keeps
the initial run fast. Widen it for a longer earnings-event history; already-cached months won't be
re-downloaded.


In [ ]:
USER_AGENT = "flightDataResearch-notebook/1.0 (personal research; contact: bruno.inzunza24@gmail.com)"

CARRIER_TO_TICKER = {"DL": "DAL", "UA": "UAL", "AA": "AAL", "WN": "LUV"}

BTS_URL_TMPL = (
    "https://transtats.bts.gov/PREZIP/"
    "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
)


def month_available(year, month):
    url = BTS_URL_TMPL.format(year=year, month=month)
    r = requests.head(url, headers={"User-Agent": USER_AGENT}, timeout=15)
    return r.status_code == 200


def fetch_month_flight_counts(year, month):
    """Daily mainline flight counts per carrier for one month, with local caching."""
    cache_path = os.path.join(CACHE_DIR, f"flights_{year}_{month:02d}.csv")
    if os.path.exists(cache_path):
        return pd.read_csv(cache_path, parse_dates=["FlightDate"])

    url = BTS_URL_TMPL.format(year=year, month=month)
    r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=60)
    if r.status_code != 200:
        return None

    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        csv_name = next(n for n in zf.namelist() if n.lower().endswith(".csv"))
        with zf.open(csv_name) as f:
            raw = pd.read_csv(f, usecols=["FlightDate", "Reporting_Airline"])

    mainline = raw[raw["Reporting_Airline"].isin(CARRIER_TO_TICKER)].copy()
    daily = (
        mainline.groupby(["FlightDate", "Reporting_Airline"])
        .size()
        .reset_index(name="flight_count")
    )
    daily["FlightDate"] = pd.to_datetime(daily["FlightDate"])
    daily.to_csv(cache_path, index=False)
    return daily


def latest_available_month(max_lookback=6):
    today = pd.Timestamp.today()
    for i in range(max_lookback):
        probe = (today - pd.DateOffset(months=i)).replace(day=1)
        if month_available(probe.year, probe.month):
            return probe.year, probe.month
    raise RuntimeError("No recently published BTS month found — check the URL pattern still works.")


LATEST_YEAR, LATEST_MONTH = latest_available_month()
print(f"Most recently published BTS month: {LATEST_YEAR}-{LATEST_MONTH:02d} "
      f"(today is {pd.Timestamp.today().date()}, so this dataset runs "
      f"{(pd.Timestamp.today().to_period('M') - pd.Period(f'{LATEST_YEAR}-{LATEST_MONTH:02d}')).n} "
      f"month(s) behind real time)")


In [ ]:
START_YEAR, START_MONTH = 2024, 6  # ~24 months of history by default; widen for more earnings events

months = pd.period_range(
    start=f"{START_YEAR}-{START_MONTH:02d}", end=f"{LATEST_YEAR}-{LATEST_MONTH:02d}", freq="M"
)

monthly_frames = []
for p in months:
    frame = fetch_month_flight_counts(p.year, p.month)
    if frame is None:
        print(f"  {p}: not available yet, skipping")
        continue
    monthly_frames.append(frame)
    time.sleep(0.2)

flights_long = pd.concat(monthly_frames, ignore_index=True)
flights_long["ticker"] = flights_long["Reporting_Airline"].map(CARRIER_TO_TICKER)

flights_daily = flights_long.pivot_table(
    index="FlightDate", columns="ticker", values="flight_count", aggfunc="sum"
)
flights_daily.columns = [f"{t}_flights" for t in flights_daily.columns]
flights_daily.index.name = "date"
print(f"Flight-count data: {flights_daily.index.min().date()} to {flights_daily.index.max().date()}, "
      f"{len(flights_daily)} calendar days")
flights_daily.tail()


In [ ]:
import yfinance as yf

TICKERS = {"DAL": "DAL", "UAL": "UAL", "AAL": "AAL", "LUV": "LUV", "SPX": "^GSPC"}

stock_raw = yf.download(
    list(TICKERS.values()),
    start=flights_daily.index.min(),
    auto_adjust=True,
    progress=False,
    group_by="ticker",
)

stock = {}
for name, symbol in TICKERS.items():
    d = stock_raw[symbol][["Close", "Volume"]].rename(
        columns={"Close": f"{name}_close", "Volume": f"{name}_vol"}
    )
    stock[name] = d

stock_prices = pd.concat(stock.values(), axis=1)
stock_prices.index = pd.to_datetime(stock_prices.index)
stock_prices = stock_prices.sort_index()
stock_prices.tail()


In [ ]:
AIRLINE_TICKERS = ["DAL", "UAL", "AAL", "LUV"]

earnings = {}
for t in AIRLINE_TICKERS:
    ed = yf.Ticker(t).get_earnings_dates(limit=40)
    ed = ed.dropna(subset=["Reported EPS"])  # keep only events that have actually happened
    ed.index = ed.index.tz_localize(None)
    earnings[t] = ed
    print(f"{t}: {len(ed)} historical earnings dates, {ed.index.min().date()} to {ed.index.max().date()}")


## 2. Calendar Alignment & De-Seasonalizing

Airlines fly every day of the week; stocks trade ~5 days/week — the same structural mismatch as the
companion notebooks. We roll non-trading-day flight counts forward into the next trading day (same
approach used for TSA/Wikipedia data).

Daily flight counts also have real day-of-week seasonality (a carrier's Tuesday schedule differs from
its Friday schedule). Because our roll-forward bucketing already spreads each calendar week's flights
across that week's ~5 trading-day buckets, summing five *consecutive trading-day* buckets recovers a
clean trailing 7-calendar-day total — comparing that trailing week to the trailing week five trading
days earlier gives a de-seasonalized **week-over-week growth rate**, which is what we use as the
primary "signal" series below (the same construct capacity analysts call WoW growth).


In [ ]:
full = stock_prices.join(flights_daily, how="outer").sort_index()

trading_days = pd.Index(stock_prices.dropna(how="all").index)
flight_cols = list(flights_daily.columns)

flights_full_sorted = full[flight_cols].sort_index()
bucket = pd.Series(pd.NaT, index=flights_full_sorted.index, dtype="object")
is_trading_day = bucket.index.isin(trading_days)
bucket.loc[is_trading_day] = bucket.index[is_trading_day]
bucket = bucket.bfill()

flights_rolled_fwd = flights_full_sorted.groupby(bucket).sum(min_count=1)
flights_rolled_fwd.index.name = "date"

df = stock_prices.join(flights_rolled_fwd, how="inner").dropna(
    subset=[f"{t}_close" for t in AIRLINE_TICKERS]
)

for t in AIRLINE_TICKERS:
    weekly_total = df[f"{t}_flights"].rolling(5).sum()
    df[f"{t}_flights_wow_growth"] = np.log(weekly_total).diff(5)

df.tail()


## 3. Overview: Flight Counts & Stock Price, Per Carrier

Raw daily counts (full 7-day calendar, so weekly cadence is visible) alongside each carrier's own
stock price.


In [ ]:
fig, axes = plt.subplots(len(AIRLINE_TICKERS), 1, figsize=(13, 4 * len(AIRLINE_TICKERS)), sharex=True)
for ax, t in zip(axes, AIRLINE_TICKERS):
    full[f"{t}_flights"].plot(ax=ax, color="tab:green", label=f"{t} daily flights")
    ax2 = ax.twinx()
    df[f"{t}_close"].plot(ax=ax2, color="tab:blue", alpha=0.8, label=f"{t} close")
    ax.set_title(f"{t}: Daily Mainline Flight Count vs. Stock Price")
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 4. Returns & Correlation (Own Flight Count vs. Own Stock)

Unlike the TSA/Wikipedia notebooks (one aggregate series vs. several stocks), here each carrier's
flight-count signal is compared **only against that same carrier's own stock** — this is the
"does an airline's own capacity trend explain its own stock's moves" question.


In [ ]:
rets = pd.DataFrame(index=df.index)
for t in AIRLINE_TICKERS + ["SPX"]:
    rets[f"{t}_ret"] = np.log(df[f"{t}_close"]).diff()
for t in AIRLINE_TICKERS:
    rets[f"{t}_flights_wow_growth"] = df[f"{t}_flights_wow_growth"]

rets = rets.dropna(how="all")
rets.tail()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, t in zip(axes.flat, AIRLINE_TICKERS):
    sub = rets[[f"{t}_ret", f"{t}_flights_wow_growth"]].dropna()
    corr_val = sub.corr().iloc[0, 1]
    sns.heatmap(sub.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(f"{t}: return vs. own flight WoW-growth corr = {corr_val:.3f}")
plt.tight_layout()
plt.show()


In [ ]:
window = 40
fig, ax = plt.subplots()
for t in AIRLINE_TICKERS:
    roll = rets[f"{t}_ret"].rolling(window).corr(rets[f"{t}_flights_wow_growth"])
    roll.plot(ax=ax, label=t)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title(f"Rolling {window}d Correlation: Own Stock Return vs. Own Flight WoW-Growth")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Lead-Lag & Granger Causality

Given the ~2-3 month BTS publication lag, a "flight data leads stock" finding here validates that
the underlying economic relationship exists — but isn't directly tradeable through *this* free feed,
since you wouldn't see the data until months later. A "stock leads flight data" finding is more
immediately informative: it says the market is already pricing in capacity changes ahead of BTS
confirming them, i.e. the market is efficient with respect to this particular information.


In [ ]:
def lagged_corr(y, x, max_lag=10):
    """Positive lag = x's past leads y's present."""
    return pd.Series({lag: y.corr(x.shift(lag)) for lag in range(-max_lag, max_lag + 1)})

max_lag = 10
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, t in zip(axes.flat, AIRLINE_TICKERS):
    sub = rets[[f"{t}_ret", f"{t}_flights_wow_growth"]].dropna()
    ll = lagged_corr(sub[f"{t}_ret"], sub[f"{t}_flights_wow_growth"], max_lag)
    colors = ["#d62728" if l == 0 else "#1f77b4" for l in ll.index]
    ll.plot(kind="bar", ax=ax, color=colors)
    ax.set_title(f"{t}: own flight WoW-growth (lag) vs return (t)")
    ax.set_xlabel("Lag (positive = flight data leads stock)")
plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

for t in AIRLINE_TICKERS:
    print(f"\n=== {t}: does own flight WoW-growth Granger-cause own return? ===")
    sub = rets[[f"{t}_ret", f"{t}_flights_wow_growth"]].dropna()
    try:
        grangercausalitytests(sub, maxlag=3, verbose=True)
    except Exception as e:
        print(f"  skipped ({e})")


## 6. Per-Earnings-Event Analysis

For each historical earnings date, we compute two different flight-count predictors and compare
each to (a) the reported EPS surprise and (b) the stock's return around the announcement. The
distinction between them matters:

- **`contemporaneous_qtr_flights_yoy`**: total mainline flights the carrier operated during the
  *fiscal quarter being reported*, vs. the same quarter one year earlier. Tests whether flight
  volume "explains" what the quarter's results turned out to be — useful for understanding the
  relationship, but **not tradeable in advance** since this data wouldn't be published until well
  after the earnings call.
- **`last_published_flights_yoy`**: the YoY flight-count growth from the most recent month that was
  **already published** as of the earnings date (i.e., what someone using only this free dataset
  could actually have known beforehand). Given the reporting lag, this typically reflects flight
  trends from a quarter or more before the one being reported — a genuinely leading (if stale)
  indicator, not a same-quarter one.

Sample sizes here are small (roughly one observation per carrier per quarter, bounded by how much
flight history we pulled) — treat results as exploratory.


In [ ]:
def event_trading_day(ts):
    """yfinance earnings timestamps carry a time-of-day; before ~noon usually means
    pre-market/BMO (event priced in same day), otherwise treat as AMC (priced in next trading day)."""
    ts = pd.Timestamp(ts).tz_localize(None)
    target = ts.normalize() if ts.hour < 12 else ts.normalize() + pd.Timedelta(days=1)
    idx = df.index[df.index >= target]
    return idx[0] if len(idx) else pd.NaT

FLIGHTS_MIN_DATE = flights_daily.index.min()
FLIGHTS_MAX_DATE = flights_daily.index.max()


def period_fully_covered(period_end_date, freq):
    """True only if our raw daily flight data spans the ENTIRE period (not just
    if the period happens to appear in a resampled index, which resample() will
    produce even from a single partial day at the start/end of our data range —
    that silently understates the period's true total and corrupts YoY ratios."""
    period = pd.Period(period_end_date, freq=freq)
    return period.start_time.normalize() >= FLIGHTS_MIN_DATE and period.end_time.normalize() <= FLIGHTS_MAX_DATE


quarterly_flights = {t: full[f"{t}_flights"].resample("QE").sum() for t in AIRLINE_TICKERS}
monthly_flights = {t: full[f"{t}_flights"].resample("ME").sum() for t in AIRLINE_TICKERS}

event_rows = []
for t in AIRLINE_TICKERS:
    q_series = quarterly_flights[t]
    m_series = monthly_flights[t]
    for earn_date, row in earnings[t].iterrows():
        q_end = (pd.Period(earn_date, freq="Q") - 1).end_time.normalize()  # quarter being reported
        q_end_prior_year = q_end - pd.DateOffset(years=1)

        if period_fully_covered(q_end, "Q") and period_fully_covered(q_end_prior_year, "Q"):
            contemporaneous_yoy = q_series[q_end] / q_series[q_end_prior_year] - 1
        else:
            contemporaneous_yoy = np.nan  # e.g. the just-reported quarter isn't fully published yet

        as_of_month_end = (earn_date.to_period("M") - 3).end_time.normalize()  # ~1 lag-quarter back
        prior_year_month_end = as_of_month_end - pd.DateOffset(years=1)
        if period_fully_covered(as_of_month_end, "M") and period_fully_covered(prior_year_month_end, "M"):
            last_published_yoy = m_series[as_of_month_end] / m_series[prior_year_month_end] - 1
        else:
            last_published_yoy = np.nan

        if pd.isna(contemporaneous_yoy) and pd.isna(last_published_yoy):
            continue  # this earnings date falls outside our flight-data window entirely

        evt_day = event_trading_day(earn_date)
        if pd.isna(evt_day):
            continue
        px = df[f"{t}_close"]
        loc = px.index.get_loc(evt_day)
        ret_1d = np.log(px.iloc[loc] / px.iloc[loc - 1]) if loc >= 1 else np.nan
        ret_3d = np.log(px.iloc[min(loc + 2, len(px) - 1)] / px.iloc[loc - 1]) if loc >= 1 else np.nan

        event_rows.append({
            "ticker": t,
            "earnings_date": earn_date,
            "reported_quarter_end": q_end,
            "eps_surprise_pct": row["Surprise(%)"],
            "contemporaneous_qtr_flights_yoy": contemporaneous_yoy,
            "last_published_flights_yoy": last_published_yoy,
            "event_ret_1d": ret_1d,
            "event_ret_3d": ret_3d,
        })

events_df = pd.DataFrame(event_rows).sort_values(["ticker", "earnings_date"])
print(f"Earnings events processed: {len(events_df)}")
print(f"  with contemporaneous_qtr_flights_yoy available: {events_df['contemporaneous_qtr_flights_yoy'].notna().sum()}")
print(f"  with last_published_flights_yoy available: {events_df['last_published_flights_yoy'].notna().sum()}")
events_df


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

pairs = [
    ("contemporaneous_qtr_flights_yoy", "eps_surprise_pct", axes[0, 0]),
    ("contemporaneous_qtr_flights_yoy", "event_ret_3d", axes[0, 1]),
    ("last_published_flights_yoy", "eps_surprise_pct", axes[1, 0]),
    ("last_published_flights_yoy", "event_ret_3d", axes[1, 1]),
]
for x_col, y_col, ax in pairs:
    sub = events_df[[x_col, y_col]].dropna()
    if len(sub) >= 3:
        corr_val = sub[x_col].corr(sub[y_col])
        ax.scatter(sub[x_col], sub[y_col], alpha=0.6)
        ax.set_title(f"{x_col}\nvs {y_col} (n={len(sub)}, corr={corr_val:.2f})", fontsize=9)
    else:
        ax.set_title(f"{x_col} vs {y_col}: not enough data (n={len(sub)})", fontsize=9)
    ax.set_xlabel(x_col, fontsize=8)
    ax.set_ylabel(y_col, fontsize=8)
plt.tight_layout()
plt.show()


## 7. Simple Signal Check: Trade the Earnings Reaction on Already-Published Flight Data

For each earnings event, go long if `last_published_flights_yoy` was positive heading into the
print, flat/short if negative — using only the tradeable (already-published) predictor — and look
at the average 3-day post-earnings return versus simply always going long earnings.

With this few events, treat this as a difference-in-means check, not a real backtest with a return
series or Sharpe ratio.


In [ ]:
signal_events = events_df.dropna(subset=["last_published_flights_yoy", "event_ret_3d"]).copy()
signal_events["signal"] = np.sign(signal_events["last_published_flights_yoy"])

summary = signal_events.groupby("signal")["event_ret_3d"].agg(["mean", "std", "count"])
always_long = signal_events["event_ret_3d"].agg(["mean", "std", "count"])

print("Mean 3-day post-earnings return, grouped by sign of already-published flight YoY growth:")
print(summary.round(4))
print(f"\nMean 3-day post-earnings return, always long (n={int(always_long['count'])}): "
      f"{always_long['mean']:.4f}")


## 8. Takeaways & Next Steps

- **Widen the history**: the default 24-month flight-data window likely gives only a handful of
  usable earnings events per carrier (Section 6 prints the exact count). Increase `START_YEAR`/
  `START_MONTH` in Section 1 and re-run — already-cached months won't be re-downloaded — to get a
  statistically meaningful sample before trusting Section 6/7's results at all.
- **Mainline-vs-total-network caveat matters here specifically**: Southwest's flight count is ~its
  whole network; American/Delta/United's isn't. If you want true total-network capacity trends for
  those three, you'd need to also pull and add each carrier's regional/codeshare partners' flight
  counts (Envoy/PSA/Piedmont for AA, Endeavor for DL, and United's several regional partners), which
  BTS reports under separate carrier codes.
- **The lag is the whole story for tradeability**: if Section 5 shows flight data leading stock
  returns, that's a real relationship this notebook cannot exploit in real time (by the time you'd
  see it, it's stale) — it's evidence for why funds pay for real-time ADS-B-based flight tracking
  (Cirium, FlightAware, OAG) instead of relying on BTS's official-but-delayed release.
- **Section 6/7's two predictors answer different questions**: `contemporaneous_qtr_flights_yoy`
  tells you whether flight volume explains a quarter's results after the fact (useful for building
  intuition); `last_published_flights_yoy` tells you what was actually knowable in advance — only
  the second one is a legitimate basis for a trading signal.
- **Robustness**: as with the companion notebooks, validate out-of-sample and account for
  transaction costs before treating anything here as an actual edge.
